In [1]:
# --- Cell 0b: Kaggle Notebook setup ---
import os

dataset_path = "/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection"
for root, dirs, files in os.walk(dataset_path):
    print(root, dirs, files[:5])
    break

def find_farm_c_dir(root):
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == "Wind Farm C" and "datasets" in dirnames:
            return dirpath
        if "datasets" in dirnames and any(f.lower() == "event_info.csv" or
                                           f.lower() == "comma_event_info.csv" for f in filenames):
            return dirpath
    raise FileNotFoundError(f"Could not locate 'Wind Farm C' data folder under {root}")

FARM_C_DIR = find_farm_c_dir(dataset_path)
print("Using Wind Farm C directory:", FARM_C_DIR)

/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection ['Wind Farm C', 'Wind Farm A', 'Wind Farm B'] ['README.md', 'README.txt']
Using Wind Farm C directory: /kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C


In [2]:
# --- Cell 1: locate the datasets dir + event/feature info files ---

import pandas as pd

def first_existing(*candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(candidates)

datasets_dir = os.path.join(FARM_C_DIR, "datasets")
event_info_path = first_existing(
    os.path.join(FARM_C_DIR, "comma_event_info.csv"),
    os.path.join(FARM_C_DIR, "event_info.csv"),
)
feature_desc_path = first_existing(
    os.path.join(FARM_C_DIR, "comma_feature_description.csv"),
    os.path.join(FARM_C_DIR, "feature_description.csv"),
)

event_info = pd.read_csv(event_info_path)
feature_description = pd.read_csv(feature_desc_path)

event_info["event_start"] = pd.to_datetime(event_info["event_start"], errors="coerce")
event_info["event_end"] = pd.to_datetime(event_info["event_end"], errors="coerce")

all_csvs = [f for f in os.listdir(datasets_dir) if f.endswith(".csv")]
comma_variant_present = any(f.startswith("comma_") for f in all_csvs)
plain_variant_present = any(not f.startswith("comma_") for f in all_csvs)

import re
def extract_dataset_id(fname):
    """Robustly pull the numeric dataset id out of '1.csv', 'comma_1.csv', etc."""
    match = re.search(r"\d+", fname)
    if not match:
        raise ValueError(f"Could not extract a numeric id from filename: {fname}")
    return int(match.group())

if comma_variant_present and plain_variant_present:
    # Same underlying data twice: "comma_<id>.csv" (comma-delimited) and "<id>.csv"
    # (semicolon-delimited, same convention as comma_event_info.csv vs event_info.csv).
    # Pick ONE variant consistently so ids aren't loaded/duplicated twice.
    USE_COMMA_VARIANT = True  # set False to use the plain "<id>.csv" / ";" files instead

    if USE_COMMA_VARIANT:
        csv_files = [f for f in all_csvs if f.startswith("comma_")]
        DATA_SEP = ","
    else:
        csv_files = [f for f in all_csvs if not f.startswith("comma_")]
        DATA_SEP = ";"
else:
    # Only one variant present.
    csv_files = all_csvs
    DATA_SEP = "," if comma_variant_present else ";"

csv_files = sorted(csv_files, key=extract_dataset_id)

# Sanity check: ids should be unique after filtering (no duplicates left in the chosen variant).
ids = [extract_dataset_id(f) for f in csv_files]
assert len(ids) == len(set(ids)), "Duplicate dataset ids remain after filtering — check the filter logic."

print(f"Using {'comma-delimited' if DATA_SEP == ',' else 'semicolon-delimited'} variant, "
      f"{len(csv_files)} files (deduplicated).")

n_anomaly = (event_info["event_label"] == "anomaly").sum()
n_normal = (event_info["event_label"] == "normal").sum()
print(f"{len(csv_files)} per-turbine files found; event_info reports "
      f"{n_anomaly} anomaly / {n_normal} normal (paper: 27 / 31).")

Using comma-delimited variant, 58 files (deduplicated).
58 per-turbine files found; event_info reports 27 anomaly / 31 normal (paper: 27 / 31).


In [3]:
# --- Cell 2: identify power & wind-speed sensor base names from feature_description.csv ---
def base_sensor_names(feature_description):
    names = feature_description["sensor_name"]
    power_names = names[names.str.startswith("power_")].tolist()
    reactive_names = names[names.str.startswith("reactive_power_")].tolist()
    wind_speed_names = names[names.str.startswith("wind_speed_")].tolist()
    return power_names, reactive_names, wind_speed_names

POWER_BASE_NAMES, REACTIVE_POWER_BASE_NAMES, WIND_SPEED_BASE_NAMES = base_sensor_names(feature_description)
print(f"Power sensors: {POWER_BASE_NAMES}")
print(f"Reactive power sensors (excluded): {REACTIVE_POWER_BASE_NAMES}")
print(f"Wind speed sensors: {WIND_SPEED_BASE_NAMES}")

Power sensors: ['power_2', 'power_5', 'power_6', 'power_17']
Reactive power sensors (excluded): ['reactive_power_119', 'reactive_power_120', 'reactive_power_121', 'reactive_power_122']
Wind speed sensors: ['wind_speed_235', 'wind_speed_236', 'wind_speed_237']


In [4]:
# --- Cell 3: peek at one data file to confirm actual column suffixes present ---
sample_df = pd.read_csv(os.path.join(datasets_dir, csv_files[0]), nrows=5)
DESCRIPTIVE_COLS = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]
SENSOR_COLS = [c for c in sample_df.columns if c not in DESCRIPTIVE_COLS]
print(f"{len(SENSOR_COLS)} sensor feature columns (expect ~952 = 957 - 5 descriptive columns).")

# Pick one usable power column (prefer "_avg") and one wind-speed column for the training filter.
def pick_avg_column(base_names, sensor_cols):
    for base in base_names:
        avg_col = f"{base}_avg"
        if avg_col in sensor_cols:
            return avg_col
    # fallback: any column starting with the base name
    for base in base_names:
        matches = [c for c in sensor_cols if c.startswith(base)]
        if matches:
            return matches[0]
    return None

POWER_COL = pick_avg_column(POWER_BASE_NAMES, SENSOR_COLS)
WIND_SPEED_COL = pick_avg_column(WIND_SPEED_BASE_NAMES, SENSOR_COLS)
print(f"Power column used for the training filter: {POWER_COL}")
print(f"Wind speed column used for the training filter: {WIND_SPEED_COL}")

952 sensor feature columns (expect ~952 = 957 - 5 descriptive columns).
Power column used for the training filter: power_2_avg
Wind speed column used for the training filter: wind_speed_235_avg


In [5]:
# --- Cell 4: Table 2 status mapping + additional filter ---
NORMAL_STATUS_IDS = {0, 2}

# Power is scaled by rated power per the paper's anonymization (Section 3.1.4), so it should be
# roughly in [0, 1]-ish range (or similar per-turbine-normalized units) rather than raw kW.
# These are generic defaults; loosen/tighten if inspection of POWER_COL / WIND_SPEED_COL suggests
# different scaling for this dataset.
WIND_SPEED_CUTIN = 3.0
WIND_SPEED_RATED = 12.0
POWER_NEAR_ZERO_THRESHOLD = 0.02

def compute_normal_label(df, power_col, wind_col):
    """Boolean Series: True = considered normal behavior for training purposes."""
    status_normal = df["status_type_id"].isin(NORMAL_STATUS_IDS)

    if power_col is not None and wind_col is not None and power_col in df.columns and wind_col in df.columns:
        wind_in_range = df[wind_col].between(WIND_SPEED_CUTIN, WIND_SPEED_RATED)
        power_near_zero = df[power_col].abs() <= POWER_NEAR_ZERO_THRESHOLD
        implausible = wind_in_range & power_near_zero
        normal = status_normal & (~implausible)
    else:
        normal = status_normal

    return normal

In [6]:
# --- Cell 5: shared config + memory helpers ---
import gc
import numpy as np

SHARD_DIR = "/tmp/care_farm_c_shards"
os.makedirs(SHARD_DIR, exist_ok=True)

SENSOR_DTYPE = np.float32

def read_dataset_csv(csv_path):
    """Read one dataset file with float32 sensor columns to halve memory vs. pandas' default."""
    dtype_map = {c: SENSOR_DTYPE for c in SENSOR_COLS}
    df = pd.read_csv(csv_path, sep=DATA_SEP, dtype=dtype_map)
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])
    df = df.sort_values("time_stamp").reset_index(drop=True)
    return df

def get_event_row(event_info, dataset_id):
    row = event_info[event_info["event_id"].astype(str) == str(dataset_id)]
    if len(row) == 0:
        raise ValueError(f"No event_info row found for dataset id {dataset_id}")
    return row.iloc[0]

In [7]:
# --- Cell 6: Pass 1 — shard normal training data to disk, fit scaler incrementally ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
shard_paths = []
shard_row_counts = []
dataset_meta = {}  # dataset_id -> {"is_anomaly_event": bool, "event_start":, "event_end":}

for fname in csv_files:
    dataset_id = extract_dataset_id(fname)
    path = os.path.join(datasets_dir, fname)

    df = read_dataset_csv(path)
    row = get_event_row(event_info, dataset_id)
    is_anomaly_event = row["event_label"] == "anomaly"
    dataset_meta[dataset_id] = {
        "is_anomaly_event": bool(is_anomaly_event),
        "event_start": row["event_start"] if is_anomaly_event else None,
        "event_end": row["event_end"] if is_anomaly_event else None,
    }

    train_mask = (df["train_test"] == "train").values
    normal_mask = compute_normal_label(df, POWER_COL, WIND_SPEED_COL).values
    train_normal_mask = train_mask & normal_mask

    normal_arr = df.loc[train_normal_mask, SENSOR_COLS].fillna(0.0).to_numpy(dtype=SENSOR_DTYPE)

    if normal_arr.shape[0] > 0:
        scaler.partial_fit(normal_arr)
        shard_path = os.path.join(SHARD_DIR, f"{dataset_id}.npy")
        np.save(shard_path, normal_arr)
        shard_paths.append(shard_path)
        shard_row_counts.append(normal_arr.shape[0])

    print(f"[{dataset_id}] normal-training rows: {normal_arr.shape[0]}")

    # Free this file's memory before moving to the next one.
    del df, normal_arr, train_mask, normal_mask, train_normal_mask
    gc.collect()

total_rows = sum(shard_row_counts)
print(f"\nTotal normal-behavior training rows across {len(shard_paths)} shards: {total_rows}")
print(f"Scaler fitted on {int(scaler.n_samples_seen_)} rows.")

[1] normal-training rows: 41700
[4] normal-training rows: 44200
[5] normal-training rows: 43917
[6] normal-training rows: 41920
[8] normal-training rows: 47170
[9] normal-training rows: 45308
[11] normal-training rows: 42080
[12] normal-training rows: 47285
[15] normal-training rows: 45019
[16] normal-training rows: 41173
[18] normal-training rows: 44244
[20] normal-training rows: 44588
[28] normal-training rows: 43247
[29] normal-training rows: 37969
[30] normal-training rows: 46183
[31] normal-training rows: 45253
[32] normal-training rows: 44237
[33] normal-training rows: 45719
[35] normal-training rows: 42789
[36] normal-training rows: 44749
[37] normal-training rows: 39995
[39] normal-training rows: 41569
[41] normal-training rows: 43763
[43] normal-training rows: 40661
[44] normal-training rows: 46160
[46] normal-training rows: 47489
[47] normal-training rows: 44921
[48] normal-training rows: 36454
[49] normal-training rows: 43643
[50] normal-training rows: 46524
[54] normal-trai

In [8]:
# --- Cell 7: split shards 75/25 and build a streaming, cross-turbine-mixed dataset ---
import tensorflow as tf

rng = np.random.default_rng(42)
order = rng.permutation(len(shard_paths))
cum_rows = np.cumsum(np.array(shard_row_counts)[order])
split_point = np.searchsorted(cum_rows, total_rows * 0.75)

train_shard_paths = [shard_paths[i] for i in order[:split_point]]
val_shard_paths = [shard_paths[i] for i in order[split_point:]]

train_rows = sum(shard_row_counts[i] for i in order[:split_point])
val_rows = total_rows - train_rows
print(f"Train rows: {train_rows} across {len(train_shard_paths)} shards")
print(f"Val rows:   {val_rows} across {len(val_shard_paths)} shards")

scaler_mean = scaler.mean_.astype(np.float32)
scaler_scale = scaler.scale_.astype(np.float32)
N_FEATURES = len(SENSOR_COLS)

def round_robin_shard_generator(paths, cycle_length=8, seed=None):
    """Yields rows drawn from `cycle_length` shards open at once, picked in random order,
    refilling from the remaining shard pool as each one is exhausted. Never holds more than
    `cycle_length` shards in memory simultaneously."""
    gen_rng = np.random.default_rng(seed)
    remaining = list(paths)
    gen_rng.shuffle(remaining)
    path_iter = iter(remaining)

    def load_next():
        try:
            p = next(path_iter)
        except StopIteration:
            return None
        arr = np.load(p)
        arr = (arr - scaler_mean) / scaler_scale
        gen_rng.shuffle(arr)  # mix row order within the shard too
        return iter(arr)

    active = [it for _ in range(cycle_length) if (it := load_next()) is not None]

    while active:
        i = gen_rng.integers(0, len(active))
        try:
            yield next(active[i])
        except StopIteration:
            new_it = load_next()
            if new_it is not None:
                active[i] = new_it
            else:
                active.pop(i)

def make_dataset(paths, batch_size, shuffle_buffer, repeat=True, cycle_length=8):
    def gen():
        seed = np.random.randint(0, 2**31 - 1)  # different shard order each repeat() cycle
        yield from round_robin_shard_generator(paths, cycle_length=cycle_length, seed=seed)

    ds = tf.data.Dataset.from_generator(
        gen, output_signature=tf.TensorSpec(shape=(N_FEATURES,), dtype=tf.float32)
    )
    if repeat:
        ds = ds.repeat()  # re-invokes gen() each pass -> re-reads shards; required because
                           # from_generator otherwise yields the corpus exactly once
    if shuffle_buffer:
        ds = ds.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.map(lambda x: (x, x))  # autoencoder target = input
    return ds.prefetch(tf.data.AUTOTUNE)

BATCH_SIZE = 64  # Table 3
train_ds = make_dataset(train_shard_paths, BATCH_SIZE, shuffle_buffer=20000, repeat=True, cycle_length=8)
val_ds = make_dataset(val_shard_paths, BATCH_SIZE, shuffle_buffer=0, repeat=True, cycle_length=8)

Train rows: 1899316 across 43 shards
Val rows:   657244 across 15 shards


I0000 00:00:1785645573.444482      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785645573.447393      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [9]:
# --- Cell 8: build & train the autoencoder ---
from tensorflow.keras import layers, models, optimizers, callbacks

N_FEATURES = len(SENSOR_COLS)

HIDDEN_LAYERS = [133, 83, 20, 83, 133]   # Table 3, Wind Farm C
LEARNING_RATE = 0.0056
NOISE_STD = 0.0                          # no denoising noise for Farm C
MAX_EPOCHS = 200
EARLY_STOP_PATIENCE = 3
STEPS_PER_EPOCH = max(1, train_rows // BATCH_SIZE)
VALIDATION_STEPS = max(1, val_rows // BATCH_SIZE)

def build_autoencoder(n_features, hidden_layers, noise_std, learning_rate):
    inputs = layers.Input(shape=(n_features,))
    x = inputs
    if noise_std > 0:
        x = layers.GaussianNoise(noise_std)(x)

    bottleneck_idx = len(hidden_layers) // 2
    for units in hidden_layers[: bottleneck_idx + 1]:
        x = layers.Dense(units, activation="relu")(x)
    for units in hidden_layers[bottleneck_idx + 1 :]:
        x = layers.Dense(units, activation="relu")(x)

    outputs = layers.Dense(n_features, activation="linear")(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=optimizers.Adam(learning_rate=learning_rate), loss="mse")
    return model

ae_model = build_autoencoder(N_FEATURES, HIDDEN_LAYERS, NOISE_STD, LEARNING_RATE)
ae_model.summary()

early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=EARLY_STOP_PATIENCE, restore_best_weights=True
)

history = ae_model.fit(
    train_ds,
    validation_data=val_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    epochs=MAX_EPOCHS,
    callbacks=[early_stop],
    verbose=2,
)

# Shards are no longer needed once the model is trained.
import shutil
shutil.rmtree(SHARD_DIR, ignore_errors=True)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 952)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 133)            │       126,749 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 83)             │        11,122 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │         1,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 83)             │         1,743 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 133)            │        11,172 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 952)            │       127,568 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 280,034 (1.07 MB)

 Trainable params: 280,034 (1.07 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200


I0000 00:00:1785645587.558402      70 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


29676/29676 - 598s - 20ms/step - loss: 0.4524 - val_loss: 0.7614
Epoch 2/200
29676/29676 - 588s - 20ms/step - loss: 0.4221 - val_loss: 1.2323
Epoch 3/200
29676/29676 - 582s - 20ms/step - loss: 0.4990 - val_loss: 0.7537
Epoch 4/200
29676/29676 - 561s - 19ms/step - loss: 0.5016 - val_loss: 0.8193
Epoch 5/200
29676/29676 - 545s - 18ms/step - loss: 0.5097 - val_loss: 1.1311
Epoch 6/200
29676/29676 - 547s - 18ms/step - loss: 0.6211 - val_loss: 0.8472


In [10]:
# --- Cell 8b: reconstruction error helper (small batches; never the whole file at once) ---
def reconstruction_error(model, X, batch_size=2048):
    X_hat = model.predict(X, batch_size=batch_size, verbose=0)
    return np.linalg.norm(X - X_hat, axis=1)  # L2 norm per row

In [11]:
# --- Cell 9: Pass 2 — single scan producing calibration data + lightweight prediction records ---
calib_scores_chunks = []
calib_is_normal_chunks = []

prediction_records = {}  # dataset_id -> dict(times, normal_mask, scores)

for fname in csv_files:
    dataset_id = extract_dataset_id(fname)
    path = os.path.join(datasets_dir, fname)

    df = read_dataset_csv(path)
    train_mask = (df["train_test"] == "train").values
    pred_mask = (df["train_test"] == "prediction").values
    normal_mask = compute_normal_label(df, POWER_COL, WIND_SPEED_COL).values

    # --- training-year rows: score for threshold calibration only (label = status normal/abnormal)
    X_train_year = df.loc[train_mask, SENSOR_COLS].fillna(0.0).to_numpy(dtype=SENSOR_DTYPE)
    X_train_year = (X_train_year - scaler_mean) / scaler_scale
    train_scores = reconstruction_error(ae_model, X_train_year)
    calib_scores_chunks.append(train_scores.astype(np.float32))
    calib_is_normal_chunks.append(normal_mask[train_mask])

    # --- prediction-window rows: keep only what CARE scoring needs, not the sensor matrix
    X_pred = df.loc[pred_mask, SENSOR_COLS].fillna(0.0).to_numpy(dtype=SENSOR_DTYPE)
    X_pred = (X_pred - scaler_mean) / scaler_scale
    pred_scores = reconstruction_error(ae_model, X_pred)

    prediction_records[dataset_id] = {
        "time_stamp": df.loc[pred_mask, "time_stamp"].to_numpy(),
        "normal_mask": normal_mask[pred_mask],
        "scores": pred_scores.astype(np.float32),
    }

    print(f"[{dataset_id}] calibration rows: {X_train_year.shape[0]}, prediction rows: {X_pred.shape[0]}")

    del df, X_train_year, X_pred, train_mask, pred_mask, normal_mask
    gc.collect()

calib_scores = np.concatenate(calib_scores_chunks)
calib_is_normal = np.concatenate(calib_is_normal_chunks)
del calib_scores_chunks, calib_is_normal_chunks
gc.collect()

print(f"\nCalibration set: {len(calib_scores)} rows "
      f"({int(calib_is_normal.sum())} normal / {int((~calib_is_normal).sum())} abnormal).")

[1] calibration rows: 51408, prediction rows: 2161
[4] calibration rows: 52560, prediction rows: 3889
[5] calibration rows: 52128, prediction rows: 667
[6] calibration rows: 52560, prediction rows: 2305
[8] calibration rows: 52560, prediction rows: 2242
[9] calibration rows: 52560, prediction rows: 3469
[11] calibration rows: 52416, prediction rows: 4021
[12] calibration rows: 52560, prediction rows: 3547
[15] calibration rows: 51696, prediction rows: 2737
[16] calibration rows: 51264, prediction rows: 2304
[18] calibration rows: 51120, prediction rows: 1728
[20] calibration rows: 51408, prediction rows: 2593
[28] calibration rows: 52560, prediction rows: 3358
[29] calibration rows: 52560, prediction rows: 2305
[30] calibration rows: 52560, prediction rows: 3551
[31] calibration rows: 52560, prediction rows: 2029
[32] calibration rows: 52560, prediction rows: 2449
[33] calibration rows: 52560, prediction rows: 3313
[35] calibration rows: 51408, prediction rows: 1207
[36] calibration ro

In [12]:
# --- Cell 10: threshold calibration for Wind Farm C (fixed threshold, Section 4.2) ---
def fbeta_from_counts(tp, fp, fn, beta=0.5):
    num = (1 + beta**2) * tp
    den = (1 + beta**2) * tp + (beta**2) * fn + fp
    return num / den if den > 0 else 0.0

def calibrate_fixed_threshold(scores, ground_truth_normal, beta=0.5, n_thresholds=200):
    candidate_thresholds = np.quantile(scores, np.linspace(0.01, 0.99, n_thresholds))

    best_threshold, best_f = None, -1.0
    for t in candidate_thresholds:
        pred_anomaly = scores > t
        pred_normal = ~pred_anomaly
        tp = np.sum(pred_anomaly & (~ground_truth_normal))
        fp = np.sum(pred_anomaly & ground_truth_normal)
        fn = np.sum(pred_normal & (~ground_truth_normal))
        f = fbeta_from_counts(tp, fp, fn, beta=beta)
        if f > best_f:
            best_f, best_threshold = f, t

    return best_threshold, best_f

THRESHOLD, calib_f_half = calibrate_fixed_threshold(calib_scores, calib_is_normal, beta=0.5)
print(f"Calibrated fixed threshold: {THRESHOLD:.4f} (calibration F½ = {calib_f_half:.3f})")

del calib_scores, calib_is_normal
gc.collect()

Calibrated fixed threshold: 50.4485 (calibration F½ = 0.820)


0

In [13]:
# --- Cell 11: turn scores into 0/1 predictions now that THRESHOLD is known ---
for dataset_id, rec in prediction_records.items():
    rec["prediction"] = (rec["scores"] > THRESHOLD).astype(int)
print(f"Applied threshold to {len(prediction_records)} datasets' prediction windows.")

Applied threshold to 58 datasets' prediction windows.


In [14]:
# --- Cell 12: CARE score components ---
BETA = 0.5
CRITICALITY_THRESHOLD = 72

def build_event_ground_truth(times, event_start, event_end):
    """1 = inside the labeled anomaly event window, 0 = normal."""
    if event_start is None or event_end is None or pd.isna(event_start) or pd.isna(event_end):
        return np.zeros(len(times), dtype=int)
    times = pd.to_datetime(times)
    result = (times >= event_start) & (times <= event_end)
    return np.asarray(result).astype(int)


def coverage_fbeta(rec, ground_truth, beta=BETA):
    """Eq. 1 — pointwise Fbeta, restricted to normal-status timestamps."""
    m = rec["normal_mask"]
    g = ground_truth[m]
    p = rec["prediction"][m]
    tp = np.sum((p == 1) & (g == 1))
    fp = np.sum((p == 1) & (g == 0))
    fn = np.sum((p == 0) & (g == 1))
    return fbeta_from_counts(tp, fp, fn, beta=beta)


def accuracy_score_care(rec):
    """Eq. 2 — Acc for normal-behavior datasets, restricted to normal-status timestamps."""
    m = rec["normal_mask"]
    p = rec["prediction"][m]
    tn = np.sum(p == 0)
    fp = np.sum(p == 1)
    return tn / (tn + fp) if (tn + fp) > 0 else 1.0


def criticality(status_normal, predictions):
    """Algorithm 1."""
    crit = np.zeros(len(predictions) + 1, dtype=int)
    for i in range(1, len(predictions) + 1):
        st = status_normal[i - 1]
        pt = predictions[i - 1]
        if not st:
            crit[i] = crit[i - 1]
        elif pt == 1:
            crit[i] = crit[i - 1] + 1
        else:
            crit[i] = max(crit[i - 1] - 1, 0)
    return crit[1:]


def event_detected(rec, threshold=CRITICALITY_THRESHOLD):
    crit = criticality(rec["normal_mask"], rec["prediction"])
    return int(crit.max() >= threshold) if len(crit) > 0 else 0


def earliness_ws(rec, event_start, event_end):
    """Eq. 3 + Figure 1 weighting, only defined for anomaly-event datasets."""
    if event_start is None or event_end is None or pd.isna(event_start) or pd.isna(event_end):
        return None
    times = pd.to_datetime(rec["time_stamp"])
    mask = (times >= event_start) & (times <= event_end)
    if mask.sum() == 0:
        return None

    event_times = times[mask]
    total_span = (event_end - event_start).total_seconds()
    if total_span <= 0:
        rel_pos = np.zeros(mask.sum())
    else:
        rel_pos = (event_times - event_start).total_seconds().values / total_span

    weights = np.where(rel_pos <= 0.5, 1.0, np.clip(1.0 - (rel_pos - 0.5) / 0.5, 0.0, 1.0))
    preds = rec["prediction"][mask]
    denom = weights.sum()
    return float((weights * preds).sum() / denom) if denom > 0 else 0.0

In [15]:
# --- Cell 13: aggregate CARE score over all Wind Farm C datasets ---
def evaluate_care_score(dataset_meta, prediction_records):
    f_beta_scores, ws_scores, acc_scores = [], [], []
    event_true_labels, event_pred_labels = [], []

    for dataset_id, meta in dataset_meta.items():
        rec = prediction_records[dataset_id]
        gt = build_event_ground_truth(rec["time_stamp"], meta["event_start"], meta["event_end"])

        if meta["is_anomaly_event"]:
            f_beta_scores.append(coverage_fbeta(rec, gt))
            ws = earliness_ws(rec, meta["event_start"], meta["event_end"])
            if ws is not None:
                ws_scores.append(ws)
        else:
            acc_scores.append(accuracy_score_care(rec))

        event_true_labels.append(int(meta["is_anomaly_event"]))
        event_pred_labels.append(event_detected(rec))

    F_beta_bar = float(np.mean(f_beta_scores)) if f_beta_scores else 0.0
    WS_bar = float(np.mean(ws_scores)) if ws_scores else 0.0
    Acc_bar = float(np.mean(acc_scores)) if acc_scores else 0.0

    true_arr = np.array(event_true_labels)
    pred_arr = np.array(event_pred_labels)
    tp = np.sum((pred_arr == 1) & (true_arr == 1))
    fp = np.sum((pred_arr == 1) & (true_arr == 0))
    fn = np.sum((pred_arr == 0) & (true_arr == 1))
    EF_beta = fbeta_from_counts(tp, fp, fn, beta=BETA)

    any_event_detected = np.sum(pred_arr) > 0

    if not any_event_detected:
        care = 0.0
    elif Acc_bar < 0.5:
        care = Acc_bar
    else:
        w1, w2, w3, w4 = 1, 1, 1, 2
        care = (w1 * F_beta_bar + w2 * WS_bar + w3 * EF_beta + w4 * Acc_bar) / (w1 + w2 + w3 + w4)

    return {
        "Coverage (F_beta_bar)": F_beta_bar,
        "Earliness (WS_bar)": WS_bar,
        "Reliability (EF_beta)": EF_beta,
        "Accuracy (Acc_bar)": Acc_bar,
        "CARE": care,
    }

results = evaluate_care_score(dataset_meta, prediction_records)
for k, v in results.items():
    print(f"{k}: {v:.3f}")

Coverage (F_beta_bar): 0.217
Earliness (WS_bar): 0.348
Reliability (EF_beta): 0.490
Accuracy (Acc_bar): 0.984
CARE: 0.605
